# Fundamentos de Ingeniería de Datos
## Transformación: de EXTRACTED a TRANSFORMED

```
INGESTA            EXTRACCIÓN              TRANSFORMACIÓN (hoy)
Kaggle → raw_df1 → extracted_df1    →    diagnosticar · limpiar · cargar · validar
        (SQLite)    (SQLite)              → df_final (en memoria) + bitácora
```

**Tema 2.2 — Transformaciones:** limpieza de nulos, duplicados y texto.

### ¿Qué es la capa TRANSFORMED?

Son los datos de `extracted_df1` **limpios y listos para analizar**. Es la primera capa en la que **sí cambiamos** los datos.

| Capa | Qué contiene | En la industria (*medallion*) |
|---|---|---|
| RAW | Datos tal como llegaron | Bronze |
| EXTRACTED | Los mismos datos + metadatos de extracción | Bronze |
| **TRANSFORMED** | **Datos limpios** | **Silver** |
| SERVING | Modelo para análisis (Unidad 3) | Gold |

> **Regla de oro:** cada capa lee **solo** de la anterior. Este notebook lee `extracted_df1` (nunca `raw_df1`) en **modo solo lectura**.

### Configuración del entorno

In [ ]:
# ── Montar Google Drive ────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
sys.path.append('/content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Utilidades')

from utils import PROYECTO_BASE, DB_PATH, inspeccionar_dataset, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()


utils.py importado correctamente
Base de datos: /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Ingesta_de_Datos/Base_de_datos/base_datos.db



In [ ]:
# ── Librerías, rutas y nombres ─────────────────────────────────────────────

import sqlite3
import pandas as pd
from datetime import datetime




TABLA_EXTRACTED   = 'extracted_df1'        # origen  (SQLite)
NOMBRE_DESTINO    = 'transformed_df1'      # nombre de la capa limpia (se cargará en la siguiente parte)
VERSION_TRANSFORM = 'v1'                   # súbela si cambias las reglas de limpieza


print()
print(f'Origen : {TABLA_EXTRACTED}  (SQLite, solo lectura)')
print()


Origen : extracted_df1  (SQLite, solo lectura)



---
## PASO 1 — Leer el origen: `extracted_df1`

Leemos en **solo lectura** y separamos las columnas en dos grupos:

- **Columnas de negocio:** las del dataset (las vamos a limpiar).
- **Metadatos** (empiezan con `_`): **no se tocan**. `_row_hash` nos permitirá rastrear cada fila limpia hasta su origen (*lineage*).

In [ ]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);


  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11


In [ ]:
# ── Leer extracted_df1 en modo solo lectura ────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()



COLS_META    = [c for c in df_ext.columns if c.startswith('_')]
COLS_NEGOCIO = [c for c in df_ext.columns if not c.startswith('_')]



print()
print(f'Columnas de negocio : {COLS_NEGOCIO}')
print(f'Metadatos           : {COLS_META}')

inspeccionar_dataset(df_ext, nombre=TABLA_EXTRACTED)


Columnas de negocio : ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
Metadatos           : ['_batch_id', '_extracted_at', '_source_table', '_row_hash']

  DATASET: extracted_df1
  Filas    : 16,598
  Columnas : 15

  Tipos de datos y valores nulos:
    Rank                         int64        nulos: 0 (0.0%)
    Name                         object       nulos: 0 (0.0%)
    Platform                     object       nulos: 0 (0.0%)
    Year                         float64      nulos: 271 (1.6%)
    Genre                        object       nulos: 0 (0.0%)
    Publisher                    object       nulos: 58 (0.3%)
    NA_Sales                     float64      nulos: 0 (0.0%)
    EU_Sales                     float64      nulos: 0 (0.0%)
    JP_Sales                     float64      nulos: 0 (0.0%)
    Other_Sales                  float64      nulos: 0 (0.0%)
    Global_Sales                 float64      nu

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,1,2026-10-28T04:26:45,raw_df1,4afd4e8c1f96d38e
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1,2026-10-28T04:26:45,raw_df1,3ee71d62807f116a
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,1,2026-10-28T04:26:45,raw_df1,688fb7c1f6ec1137


---
## PASO 2 — Diagnóstico de calidad (antes)

**No se limpia lo que no se ha medido.** Antes de transformar, medimos los problemas. Al final repetimos la medición y comparamos: esa comparación va en tu **reporte de validación**.

En este notebook trabajamos tres problemas de calidad:

| Problema | Pregunta | Cómo lo medimos |
|---|---|---|
| **Nulos** (completitud) | ¿Faltan datos? | Celdas vacías por columna |
| **Repetidos** (unicidad) | ¿Hay filas duplicadas? | Filas con todas sus columnas iguales |
| **Espacios** (formato del texto) | ¿El texto está bien escrito? | Espacios al inicio, al final o dobles |

In [ ]:
# ── Reglas del dataset (ajústalas a TU dataset) ────────────────────────────

COLS_TEXTO = ['Name', 'Platform', 'Genre', 'Publisher']       # columnas de texto que vamos a revisar

In [ ]:
# ── Nulos por columna ──────────────────────────────────────────────────────

nulos = df_ext[COLS_NEGOCIO].isna().sum()

print()
print(f'Total de celdas nulas: {nulos.sum():,}')
print()

nulos


Total de celdas nulas: 329



,0
Rank,0
Name,0
Platform,0
Year,271
Genre,0
Publisher,58
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0


In [ ]:
# ── Registros repetidos ────────────────────────────────────────────────────

# Exactos: todas las columnas del dataset son iguales
dup_exactos = df_ext.duplicated(subset=COLS_NEGOCIO).sum()

print()
print(f'Duplicados exactos: {dup_exactos}')
print()

# Ver los registros repetidos
repetidos = df_ext[df_ext.duplicated(subset=COLS_NEGOCIO, keep=False)]
repetidos[COLS_NEGOCIO]


Duplicados exactos: 0



,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales


In [ ]:
# ── Espacios extra en el texto ─────────────────────────────────────────────

def tiene_espacios_extra(columna):

    '''True en cada celda con espacios al inicio, al final o dobles.'''

    limpio = columna.str.strip().str.replace(r'\s+', ' ', regex=True) # Función en padndas
    return columna.notna() & (columna != limpio)



In [ ]:

# apply() ejecuta la función en cada columna de COLS_TEXTO
con_espacios = df_ext[COLS_TEXTO].apply(tiene_espacios_extra)

print()
print(f'Celdas con espacios extra: {con_espacios.sum().sum()}')
print()

# Ver las filas que tienen al menos una celda con espacios extra
df_ext.loc[con_espacios.any(axis=1), COLS_TEXTO]


Celdas con espacios extra: 5



,Name,Platform,Genre,Publisher
3465,FIFA: Road to World Cup 98,N64,Sports,Electronic Arts
4528,Hunter: The Reckoning Wayward,PS2,Action,Interplay
10927,U-Sing 2,Wii,Misc,Mindscape
14967,Rengoku II: The Stairway To H.E.A.V.E.N.,PSP,Action,Konami Digital Entertainment
16011,Tom Clancy's Ghost Recon Advanced Warfighter ...,X360,Shooter,Ubisoft


> Para ver los espacios, compara la longitud del texto: `'Nintendo '` tiene 9 caracteres y `'Nintendo'` tiene 8. A simple vista se ven iguales, pero para pandas son **valores distintos**.

In [ ]:
# ── Resumen del diagnóstico (línea base para comparar después) ─────────────

calidad_antes = pd.Series({
    'filas'                  : len(df_ext),
    'celdas_nulas'           : nulos.sum(),
    'duplicados_exactos'     : dup_exactos,
    'celdas_con_espacios'    : con_espacios.sum().sum(),
}, name='antes')

calidad_antes.to_frame()

,antes
filas,16598
celdas_nulas,329
duplicados_exactos,0
celdas_con_espacios,5


---
## PASO 3 — Transformaciones

Ahora sí **cambiamos los datos**, atacando lo que encontró el diagnóstico del Paso 2. Aplicamos las reglas **una por una** sobre una copia (`df`) y anotamos cada una en una **bitácora**: qué se hizo y cuántas filas cambió. La bitácora es el *lineage* de las transformaciones y va tal cual en tu reporte.

| # | Transformación | ¿Qué problema resuelve? |
|---|---|---|
| 3.1 | Limpiar espacios en el texto | Espacios |
| 3.2 | Tratar nulos | Nulos |
| 3.3 | Quitar duplicados exactos | Repetidos |

> **El orden importa:** primero se limpian los espacios. Así, `'Nintendo '` y `'Nintendo'` quedan iguales **antes** de buscar duplicados, y esas filas también se detectan como repetidas.

Estas tres son la base. En tu proyecto agrega las transformaciones que **tus datos** necesiten, y registra cada una en la bitácora.

In [ ]:
# ── Bitácora de transformaciones ───────────────────────────────────────────

bitacora = []

def registrar(paso, descripcion, filas_antes, filas_despues, afectadas):
    '''Anota un paso en la bitácora y lo muestra.'''

    bitacora.append({
        'paso'            : paso,
        'descripcion'     : descripcion,
        'filas_antes'     : filas_antes,
        'filas_despues'   : filas_despues,
        'filas_afectadas' : int(afectadas),
    })

    print(f'[{paso}] {descripcion}')
    print(f'       filas: {filas_antes:,} → {filas_despues:,}   (afectadas: {int(afectadas):,})')


df = df_ext.copy()        # trabajamos sobre una copia: df_ext queda intacto para comparar

print()
print(f'Punto de partida: {len(df):,} filas')
print()


Punto de partida: 16,598 filas



### 3.1 Limpiar espacios en el texto

Quitamos los espacios al inicio y al final, y dejamos un solo espacio entre palabras. `apply()` ejecuta la función de limpieza en cada columna de `COLS_TEXTO`.

In [ ]:
# ── 3.1 Limpiar espacios ───────────────────────────────────────────────────

def limpiar_texto(columna):
    '''Quita espacios al inicio y al final, y deja un solo espacio entre palabras.'''
    return columna.str.strip().str.replace(r'\s+', ' ', regex=True)


celdas = df[COLS_TEXTO].apply(tiene_espacios_extra).sum().sum()     # cuántas se van a corregir

df[COLS_TEXTO] = df[COLS_TEXTO].apply(limpiar_texto)

registrar('3.1', f'Limpiar espacios en {COLS_TEXTO}', len(df), len(df), celdas)

[3.1] Limpiar espacios en ['Name', 'Platform', 'Genre', 'Publisher']
       filas: 16,598 → 16,598   (afectadas: 5)


### 3.2 Tratar nulos

No hay una regla única: **cada columna se decide según su significado** y la decisión se documenta.

| Columna | Opciones | Decisión | Por qué |
|---|---|---|---|
| `Publisher` | eliminar / imputar | **Imputar `'Unknown'`** | El juego existe y sus ventas son válidas; solo falta el dato de la editorial |
| `Year` | eliminar / imputar con la mediana / dejar nulo | **Eliminar la fila** | Inventar un año de lanzamiento sería un dato falso. Son pocas filas (revisa el Paso 2) |

> En tu dataset las decisiones pueden ser otras. Lo importante es **justificarlas** en el reporte.

In [ ]:
# ── 3.2 Nulos ──────────────────────────────────────────────────────────────

# Publisher → 'Unknown'
nulos_pub = df['Publisher'].isna().sum()
df['Publisher'] = df['Publisher'].fillna('Unknown')
registrar('3.2a', "Publisher nulo → 'Unknown'", len(df), len(df), nulos_pub)

# Year nulo → eliminar fila
antes = len(df)
df = df.dropna(subset=['Year'])

registrar('3.2b', 'Eliminar filas sin Year', antes, len(df), antes - len(df))

[3.2a] Publisher nulo → 'Unknown'
       filas: 16,598 → 16,598   (afectadas: 58)
[3.2b] Eliminar filas sin Year
       filas: 16,598 → 16,327   (afectadas: 271)


### 3.3 Quitar duplicados exactos

Si todas las columnas del dataset son iguales, la fila está repetida: se conserva la primera y se eliminan las demás. Comparamos solo `COLS_NEGOCIO`, porque los metadatos (`_row_hash`, `_batch_id`) siempre son distintos entre filas.

> Aunque el Paso 2 haya dado 0, la regla se deja en el pipeline: protege las siguientes ejecuciones, cuando lleguen datos nuevos.

In [ ]:
# ── 3.3 Duplicados exactos ─────────────────────────────────────────────────

antes = len(df)
df = df.drop_duplicates(subset=COLS_NEGOCIO, keep='first')

registrar('3.3', 'Eliminar duplicados exactos', antes, len(df), antes - len(df))

[3.3] Eliminar duplicados exactos
       filas: 16,327 → 16,327   (afectadas: 0)


### Preparar `df_final`

Igual que en EXTRACTED, agregamos cuándo y con qué versión de reglas se transformó cada fila. Los metadatos de extracción **se conservan**: con `_row_hash` cualquier fila limpia se rastrea hasta `extracted_df1` y `raw_df1`.

In [ ]:
df

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,1,2026-10-28T04:26:45,raw_df1,4afd4e8c1f96d38e
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1,2026-10-28T04:26:45,raw_df1,3ee71d62807f116a
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,1,2026-10-28T04:26:45,raw_df1,688fb7c1f6ec1137
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,1,2026-10-28T04:26:45,raw_df1,a60698219af7ed96
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,1,2026-10-28T04:26:45,raw_df1,2644afa96579b93b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16593,16596,Woody Woodpecker in Crazy Castle 5,GBA,2002.0,Platform,Kemco,0.01,0.00,0.00,0.00,0.01,1,2026-10-28T04:26:45,raw_df1,cfa73662bda97c4d
16594,16597,Men in Black II: Alien Escape,GC,2003.0,Shooter,Infogrames,0.01,0.00,0.00,0.00,0.01,1,2026-10-28T04:26:45,raw_df1,6704714b108341bd
16595,16598,SCORE International Baja 1000: The Official Game,PS2,2008.0,Racing,Activision,0.00,0.00,0.00,0.00,0.01,1,2026-10-28T04:26:45,raw_df1,c903fb255c6703cf
16596,16599,Know How 2,DS,2010.0,Puzzle,7G//AMES,0.00,0.01,0.00,0.00,0.01,1,2026-10-28T04:26:45,raw_df1,edad10f8defa04a8


In [ ]:
# ── Metadatos de transformación ────────────────────────────────────────────

df['_transformed_at']    = pd.Timestamp(datetime.now().replace(microsecond=0))
df['_transform_version'] = VERSION_TRANSFORM

df_final = df.reset_index(drop=True)

inspeccionar_dataset(df_final, nombre=f'{NOMBRE_DESTINO} (antes de cargar)')


  DATASET: transformed_df1 (antes de cargar)
  Filas    : 16,327
  Columnas : 17

  Tipos de datos y valores nulos:
    Rank                         int64        nulos: 0 (0.0%)
    Name                         object       nulos: 0 (0.0%)
    Platform                     object       nulos: 0 (0.0%)
    Year                         float64      nulos: 0 (0.0%)
    Genre                        object       nulos: 0 (0.0%)
    Publisher                    object       nulos: 0 (0.0%)
    NA_Sales                     float64      nulos: 0 (0.0%)
    EU_Sales                     float64      nulos: 0 (0.0%)
    JP_Sales                     float64      nulos: 0 (0.0%)
    Other_Sales                  float64      nulos: 0 (0.0%)
    Global_Sales                 float64      nulos: 0 (0.0%)
    _batch_id                    int64        nulos: 0 (0.0%)
    _extracted_at                object       nulos: 0 (0.0%)
    _source_table                object       nulos: 0 (0.0%)
    _row_hash  

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash,_transformed_at,_transform_version
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,1,2026-10-28T04:26:45,raw_df1,4afd4e8c1f96d38e,2026-10-29 22:17:32,v1
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,1,2026-10-28T04:26:45,raw_df1,3ee71d62807f116a,2026-10-29 22:17:32,v1
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,1,2026-10-28T04:26:45,raw_df1,688fb7c1f6ec1137,2026-10-29 22:17:32,v1


In [ ]:
# ── La bitácora completa ───────────────────────────────────────────────────

df_bitacora = pd.DataFrame(bitacora)
df_bitacora

,paso,descripcion,filas_antes,filas_despues,filas_afectadas
0,3.1,"Limpiar espacios en ['Name', 'Platform', 'Genr...",16598,16598,5
1,3.2a,Publisher nulo → 'Unknown',16598,16598,58
2,3.2b,Eliminar filas sin Year,16598,16327,271
3,3.3,Eliminar duplicados exactos,16327,16327,0


---
## PASO 4 — Diagnóstico de calidad (después)

Repetimos **las mismas mediciones** del Paso 2, ahora sobre `df_final`.

In [ ]:
# ── Nulos, repetidos y espacios después de transformar ─────────────────────

nulos_d        = df_final[COLS_NEGOCIO].isna().sum()
dup_exactos_d  = df_final.duplicated(subset=COLS_NEGOCIO).sum()
con_espacios_d = df_final[COLS_TEXTO].apply(tiene_espacios_extra)

print()
print(f'Total de celdas nulas     : {nulos_d.sum():,}')
print(f'Duplicados exactos        : {dup_exactos_d}')
print(f'Celdas con espacios extra : {con_espacios_d.sum().sum()}')
print()


Total de celdas nulas     : 0
Duplicados exactos        : 0
Celdas con espacios extra : 0



In [ ]:
# ── Antes vs. después ──────────────────────────────────────────────────────

calidad_despues = pd.Series({
    'filas'                  : len(df_final),
    'celdas_nulas'           : nulos_d.sum(),
    'duplicados_exactos'     : dup_exactos_d,
    'celdas_con_espacios'    : con_espacios_d.sum().sum(),
}, name='después')

df_calidad = pd.DataFrame({'antes': calidad_antes, 'después': calidad_despues})
df_calidad['cambio'] = df_calidad['después'] - df_calidad['antes']
df_calidad

,antes,después,cambio
filas,16598,16327,-271
celdas_nulas,329,0,-329
duplicados_exactos,0,0,0
celdas_con_espacios,5,0,-5


**Cómo leerla:**
- Nulos, duplicados y espacios deben quedar en **0**.
- La baja de `filas` debe coincidir con la suma de filas eliminadas en la bitácora.

In [ ]:
# ── ¿Cuadra la bitácora con el conteo de filas? ────────────────────────────

eliminadas_bitacora = (df_bitacora['filas_antes'] - df_bitacora['filas_despues']).sum()
eliminadas_real     = len(df_ext) - len(df_final)

print()
print(f'Filas eliminadas según la bitácora : {eliminadas_bitacora:,}')
print(f'Filas eliminadas en realidad       : {eliminadas_real:,}')

if eliminadas_bitacora == eliminadas_real:
    print('Cuadra')
else:
    print('NO CUADRA: revisa la bitácora')
print()


Filas eliminadas según la bitácora : 271
Filas eliminadas en realidad       : 271
Cuadra



---
## PASO 5 — Carga: escribir la capa TRANSFORMED en Parquet

Hasta aquí, `df_final` **solo vive en memoria**: si cierras Colab, se pierde. La **carga** (la "L" de ETL) lo guarda en un destino persistente para que otros, y la Unidad 3, lo consuman.

```
extracted_df1 (SQLite)  →  transformaciones  →  transformed_df1.parquet  (Drive)
```

### ¿Por qué Parquet y no el `.db`?

- En la industria, la capa limpia (*silver*) vive normalmente en archivos **Parquet** dentro de un *data lake*. La base de datos para análisis llega después, al modelar (Unidad 3).
- Parquet **guarda el tipo de cada columna** dentro del archivo. SQLite y CSV no lo conservan.



### ¿Append o sobrescribir?

| Estrategia | Qué hace | ¿Dónde la usamos? |
|---|---|---|
| **Append** | Agrega filas nuevas al final | EXTRACTED: cada lote es historia que no se puede perder |
| **Sobrescribir** (*full refresh*) | Reemplaza el archivo completo | **TRANSFORMED**: se puede recalcular entero desde EXTRACTED |

Aquí **sobrescribimos**. Si mañana cambia una regla de limpieza, basta con volver a ejecutar el notebook. A esto se le llama **idempotencia**: ejecutar el pipeline una o diez veces deja el mismo resultado.

In [ ]:
# ── Librerías y rutas de la carga ──────────────────────────────────────────

import os
import json

TRANSFORMACION_PATH = f'{PROYECTO_BASE}/Transformacion_de_Datos'
PROCESSED_PATH      = f'{TRANSFORMACION_PATH}/Processed'     # aquí se guardan los Parquet
LOGS_PATH           = f'{TRANSFORMACION_PATH}/Logs'          # log y lineage
REPORTES_PATH       = f'{TRANSFORMACION_PATH}/Reportes'      # evidencias para el reporte


os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(REPORTES_PATH, exist_ok=True)


PARQUET_DESTINO = f'{PROCESSED_PATH}/{NOMBRE_DESTINO}.parquet'
LOG_PATH        = f'{LOGS_PATH}/transformacion_log.jsonl'
LINEAGE_PATH    = f'{LOGS_PATH}/lineage_{NOMBRE_DESTINO}.json'

print()
print(f'Destino : {PARQUET_DESTINO}')
print()


Destino : /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Processed/transformed_df1.parquet



In [ ]:
# ── Log de operaciones (igual que en EXTRACCIÓN: una línea JSON por operación) ──

def escribir_log(etapa, filas_entrada, filas_salida, estado='exitoso', detalle=None):

    '''Agrega UNA línea al log de transformación.'''


    evento = {

        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'etapa'        : etapa,
        'origen'       : TABLA_EXTRACTED,
        'destino'      : PARQUET_DESTINO,
        'filas_entrada': filas_entrada,
        'filas_salida' : filas_salida,
        'version'      : VERSION_TRANSFORM,
        'estado'       : estado,
        'detalle'      : detalle,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:      # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')


print()
print('escribir_log() lista')
print()


escribir_log() lista



In [ ]:
# ── Cargar df_final a Parquet ──────────────────────────────────────────────

df_final.to_parquet(PARQUET_DESTINO, index=False)        # si el archivo existe, se sobrescribe

tamanio_mb = os.path.getsize(PARQUET_DESTINO) / 1_048_576

escribir_log('carga_parquet', len(df_ext), len(df_final), detalle=f'{tamanio_mb:.2f} MB')


print()
print(f'{TABLA_EXTRACTED} → {NOMBRE_DESTINO}.parquet')
print(f'  Filas de entrada : {len(df_ext):,}')
print(f'  Filas cargadas   : {len(df_final):,}')
print(f'  Tamaño           : {tamanio_mb:.2f} MB')
print()


extracted_df1 → transformed_df1.parquet
  Filas de entrada : 16,598
  Filas cargadas   : 16,327
  Tamaño           : 0.74 MB



### ¿Se conservaron los tipos?

Leemos el archivo de vuelta y comparamos sus tipos con los de `df_final`. Deben ser **iguales**: esa es la ventaja de Parquet.

In [ ]:
# ── Leer el Parquet de vuelta ──────────────────────────────────────────────

df_leido = pd.read_parquet(PARQUET_DESTINO)

print()
print(f'Filas leídas: {len(df_leido):,}')
print()

pd.DataFrame({'en memoria': df_final.dtypes, 'desde Parquet': df_leido.dtypes})


Filas leídas: 16,327



,en memoria,desde Parquet
Rank,int64,int64
Name,object,object
Platform,object,object
Year,float64,float64
Genre,object,object
Publisher,object,object
NA_Sales,float64,float64
EU_Sales,float64,float64
JP_Sales,float64,float64
Other_Sales,float64,float64


---
## PASO 6 — Lineage y evidencias

El **lineage** documenta el recorrido completo del dato: **origen → transformaciones → destino**. Lo guardamos como JSON junto al log. Cada fila, además, conserva su `_row_hash`, que la vincula con `extracted_df1` y `raw_df1`.

También guardamos en `Reportes/` las tablas que van en el **reporte de validación**: la bitácora y la comparación antes vs. después.

In [ ]:
# ── Guardar el lineage ─────────────────────────────────────────────────────

lineage = {

    'recorrido'        : ['Kaggle (CSV)', 'raw_df1 (SQLite)', f'{TABLA_EXTRACTED} (SQLite)', f'{NOMBRE_DESTINO} (Parquet)'],
    'origen'           : TABLA_EXTRACTED,
    'base_de_datos'    : DB_PATH,
    'filas_origen'     : len(df_ext),
    'destino'          : PARQUET_DESTINO,
    'filas_destino'    : len(df_final),
    'estrategia_carga' : 'sobrescribir (full refresh)',
    'version_reglas'   : VERSION_TRANSFORM,
    'generado_en'      : datetime.now().isoformat(timespec='seconds'),
    'trazabilidad'     : '_row_hash vincula cada fila con extracted_df1 y raw_df1',
    'transformaciones' : bitacora,                                    # la bitácora del Paso 3
    'columnas_destino' : df_final.dtypes.astype(str).to_dict(),       # {columna: tipo}
}

with open(LINEAGE_PATH, 'w', encoding='utf-8') as f:
    json.dump(lineage, f, indent=2, ensure_ascii=False, default=int)

print()
print(f'Lineage guardado en: {LINEAGE_PATH}')
print()
print(' → '.join(lineage['recorrido']))
print()


Lineage guardado en: /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Logs/lineage_transformed_df1.json

Kaggle (CSV) → raw_df1 (SQLite) → extracted_df1 (SQLite) → transformed_df1 (Parquet)



> `default=int` convierte a entero los números de pandas/NumPy (por ejemplo, un `int64`), que el módulo `json` no sabe escribir por sí solo.

In [ ]:
# ── Guardar las evidencias para el reporte ─────────────────────────────────

ruta_bitacora = f'{REPORTES_PATH}/bitacora_{NOMBRE_DESTINO}.csv'
ruta_calidad  = f'{REPORTES_PATH}/calidad_{NOMBRE_DESTINO}.csv'


df_bitacora.to_csv(ruta_bitacora, index=False, encoding='utf-8-sig')       # utf-8-sig: abre bien en Excel
df_calidad.to_csv(ruta_calidad, index_label='metrica', encoding='utf-8-sig')


print()
print('Evidencias guardadas:')
print(f'  {ruta_bitacora}')
print(f'  {ruta_calidad}')
print()




Evidencias guardadas:
  /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Reportes/bitacora_transformed_df1.csv
  /content/drive/MyDrive/CUGDL/Fundamentos_de_Ingenieria_de_Datos/Transformacion_de_Datos/Reportes/calidad_transformed_df1.csv



---
## PASO 7 — Validación con DuckDB (sobre el Parquet ya cargado)

Una carga no está terminada hasta que **se valida lo que quedó en el destino**, no lo que había en memoria. Con **DuckDB** consultamos el archivo Parquet directamente con SQL, sin cargarlo a ninguna base.



In [ ]:
# ── DuckDB ─────────────────────────────────────────────────────────────────
# En Colab suele venir instalado; si no, quita el # de la siguiente línea:
# !pip install duckdb

import duckdb

con = duckdb.connect()        # base en memoria: no crea ningún archivo

P = f"read_parquet('{PARQUET_DESTINO}')"         # así se lee un Parquet dentro de SQL

print()
print(f'DuckDB {duckdb.__version__} listo')
print()


DuckDB 1.3.2 listo



In [ ]:
# ── Primer vistazo: SQL directo sobre el archivo ───────────────────────────

con.sql(f'''
    SELECT Name, Platform, Year, Publisher, Global_Sales
    FROM {P}
    ORDER BY Global_Sales DESC
    LIMIT 5
''').df()

,Name,Platform,Year,Publisher,Global_Sales
0,Wii Sports,Wii,2006.0,Nintendo,82.74
1,Super Mario Bros.,NES,1985.0,Nintendo,40.24
2,Mario Kart Wii,Wii,2008.0,Nintendo,35.82
3,Wii Sports Resort,Wii,2009.0,Nintendo,33.00
4,Pokemon Red/Pokemon Blue,GB,1996.0,Nintendo,31.37


In [ ]:
# ── Esquema: columnas y tipos guardados en el archivo ──────────────────────

con.sql(f'DESCRIBE SELECT * FROM {P}').df()

,column_name,column_type,null,key,default,extra
0,Rank,BIGINT,YES,None,None,None
1,Name,VARCHAR,YES,None,None,None
2,Platform,VARCHAR,YES,None,None,None
3,Year,DOUBLE,YES,None,None,None
4,Genre,VARCHAR,YES,None,None,None
5,Publisher,VARCHAR,YES,None,None,None
6,NA_Sales,DOUBLE,YES,None,None,None
7,EU_Sales,DOUBLE,YES,None,None,None
8,JP_Sales,DOUBLE,YES,None,None,None
9,Other_Sales,DOUBLE,YES,None,None,None


In [ ]:
# ── Conteo y nulos en el archivo cargado ───────────────────────────────────

con.sql(f'''
    SELECT COUNT(*)                                           AS filas,
           SUM(CASE WHEN Year      IS NULL THEN 1 ELSE 0 END) AS nulos_year,
           SUM(CASE WHEN Publisher IS NULL THEN 1 ELSE 0 END) AS nulos_publisher
    FROM {P}
''').df()

,filas,nulos_year,nulos_publisher
0,16327,0.0,0.0


In [ ]:
# ── Cerrar la conexión de DuckDB ───────────────────────────────────────────

con.close()
AHR
print()
print('Conexión cerrada')
print()


Conexión cerrada

